In [ ]:
!pip install librosa

In [ ]:
!pip install matplotlib

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
from scipy.ndimage import maximum_filter

# 1. Force absolute status tracking variables
print("[1] Script started successfully.")

# Put the EXACT filename of one MP3 inside your folder here:
TARGET_FILE = "./EE200 Project Song Database/Bohemian Rhapsody.mp3"

print(f"[2] Checking target file path: '{TARGET_FILE}'")
if not os.path.exists(TARGET_FILE):
    print(f"[-] ERROR: Python cannot find the file at '{TARGET_FILE}'")
    print("    Make sure the folder name and file name are spelled exactly right.")
else:
    print(f"[+] SUCCESS: File located! Size is {os.path.getsize(TARGET_FILE)} bytes.")
    
    print("[3] Attempting to load audio array with librosa...")
    try:
        # Load just the first 30 seconds to make it process instantly
        sig, sr = librosa.load(TARGET_FILE, sr=22050, mono=True, duration=30.0)
        print(f"[+] Audio loaded successfully! Array shape: {sig.shape}, Sample Rate: {sr}")
        
        print("[4] Computing Spectral Dimensions...")
        # 1. Global DFT
        global_dft = np.fft.rfft(sig)
        dft_freqs = np.fft.rfftfreq(len(sig), d=1/sr)
        dft_magnitude = np.abs(global_dft)

        # 2. Spectrogram
        f, t, Sxx = spectrogram(sig, sr, nperseg=2048, noverlap=1024)
        Sxx_log = 10 * np.log10(Sxx + 1e-10)
        
        print("[5] Generating Matplotlib Window Framework...")
        fig, ax = plt.subplots(1, 2, figsize=(15, 5))
        
        # Left Panel: Global DFT Curve
        ax[0].plot(dft_freqs, dft_magnitude, color='purple', alpha=0.85)
        ax[0].set_title("Global DFT Magnitude Profile")
        ax[0].set_xlabel("Frequency (Hz)")
        ax[0].set_ylabel("Net Cumulative Amplitude")
        ax[0].set_xlim(0, 4000)
        ax[0].grid(True)

        # Right Panel: Spectrogram Matrix
        mesh = ax[1].pcolormesh(t, f, Sxx_log, shading='gouraud', cmap='magma')
        ax[1].set_title("STFT Spectrogram View")
        ax[1].set_xlabel("Time (Seconds)")
        ax[1].set_ylabel("Frequency (Hz)")
        ax[1].set_ylim(0, 4000)
        fig.colorbar(mesh, ax=ax[1], label="Intensity Level (dB)")

        plt.tight_layout()
        print("[6] Calling plt.show(). Plots should render below:")
        plt.show()
        print("[7] Execution pipeline finished.")
        
    except Exception as error:
        print(f"[-] CRITICAL RUNTIME ERROR: {error}")